This is a tutorial for the projected (Woodbury) likelihood of the flip package: https://github.com/corentinravoux/flip \
The dense likelihood builds and factorises the full $N \times N$ covariance at each call. \
The projected likelihood projects the covariance once on the $K$ modes with signal-to-noise above a threshold,
chosen at a fiducial point, and then only handles $K \times K$ matrices: it is faster and lighter for noisy
velocities (SNIa, Tully-Fisher, fundamental plane) when $N$ is large. \
It is an approximation, valid near the fiducial point. The data used here are a small subsample of a simulation:
the speed-up appears for $N \gtrsim 2000$.

In [1]:
import time

import numpy as np
import pandas as pd

from pathlib import Path
from flip import utils, data_vector, __flip_dir_path__
from flip.covariance import covariance, fitter
from flip.covariance.covariance import ProjectedCovariance

flip_base = Path(__flip_dir_path__)
data_path = flip_base / "data"

flip 2.1.0 -- subpackage availability
  data           OK
  data_vector    OK  [optional missing: jax (jit paths) (jax)]
  comparison     OK  [optional missing: jax (jit paths) (jax)]
  forward        UNAVAILABLE  missing required: jax, jax_cosmo
  power_spectra  OK  [optional missing: pyccl engine (pyccl); class engine (classy); cosmoprimo engine (cosmoprimo)]
  covariance     OK  [optional ok: parquet io, skgp emulator | optional missing: jax (jit paths) (jax); gp emulator (GPy); nn emulator (torch); firecrown interface (firecrown)]
JAX backend: not installed, using numpy/scipy


Loading the data, located in the package itself: SNIa with SALT2 fits

In [2]:
data_velocity = pd.read_parquet(data_path / "data_velocity.parquet")
coordinates_velocity = np.array([data_velocity["ra"], data_velocity["dec"], data_velocity["rcom_zobs"]])

DataVel = data_vector.snia_vectors.VelFromSALTfit(data_velocity.to_dict(orient="list"), velocity_estimator="full", h=0.70)

ktt, ptt = np.loadtxt(data_path / "power_spectrum_tt.txt")
sigmau_fiducial = 15
power_spectrum_dict = {"vv": [[ktt, ptt * utils.Du(ktt, sigmau_fiducial) ** 2]]}

print("Number of simulated SNIa: ", len(data_velocity["ra"]))

Number of simulated SNIa:  518


First step, as usual: the covariance matrix

In [3]:
covariance_fit = covariance.CovMatrix.init_from_flip(
    "carreres23",
    "velocity",
    power_spectrum_dict,
    coordinates_velocity=coordinates_velocity,
    size_batch=10_000,
    number_worker=16,
)

[ 000000.77 ]: 11-05 07:46  root            INFO     Covariance matrix generated from flip with carreres23 model in 4.52e-01 seconds


Second step: the projected covariance, built once from the dense covariance and the data at a fiducial point. \
`snr_min` sets the number of kept modes $K$: modes with a whitened signal-to-noise below it are dropped. \
Take the fiducial point close to the expected best fit, with a low velocity dispersion `sigv`.

In [4]:
parameter_dict = {
    "fs8": {"value": 0.4, "limit_low": 0.0, "limit_up": 1.0, "fixed": False},
    "sigv": {"value": 200, "limit_low": 0.0, "limit_up": 1000, "fixed": False},
    "alpha": {"value": 0.14, "limit_low": 0.0, "limit_up": 0.25, "fixed": False},
    "beta": {"value": 3.1, "limit_low": 2.5, "limit_up": 4.5, "fixed": False},
    "M_0": {"value": -19.0, "limit_low": -21, "limit_up": -18, "fixed": False},
    "sigma_M": {"value": 0.12, "limit_low": 0.0, "limit_up": 1.0, "fixed": False},
}
fiducial = {name: properties["value"] for name, properties in parameter_dict.items()}
fiducial["sigv"] = 150

projected_covariance = ProjectedCovariance.init_from_data(covariance_fit, DataVel, fiducial, snr_min=0.01)

print(f"Kept modes: K = {projected_covariance.k} of N = {projected_covariance.size}")

[ 000000.82 ]: 11-05 07:46  root            INFO     Projected covariance keeps 175 of 518 modes: the data are signal dominated and the projection gains little.


Kept modes: K = 175 of N = 518


Third step: the fits, dense and projected. Only the covariance and the likelihood type change.

In [5]:
minuit_dense = fitter.FitMinuit.init_from_covariance(
    covariance_fit,
    DataVel,
    parameter_dict,
    likelihood_type="multivariate_gaussian",
    likelihood_properties={"inversion_method": "cholesky"},
)
minuit_dense.run()

/global/homes/r/ravouxco/.local/perlmutter/python-3.13/26.11.0/lib/python3.13/site-packages/iminuit/util.py:133: IMinuitWarning: Assigned errors must be positive. Non-positive values are replaced by a heuristic.
  self._set(index, value)


[ 000001.42 ]: 11-05 07:46  root            INFO     ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 4088                       │              Nfcn = 273              │
│ EDM = 3.21e-05 (Goal: 0.0001)    │            time = 0.6 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬─────────┬───────────┬────────

{'fs8': 0.3746462916972586,
 'sigv': 0.3757009775682838,
 'alpha': 0.12723877225638744,
 'beta': 3.0939008520094577,
 'M_0': -19.062562789525014,
 'sigma_M': 0.1404188942907373}

In [6]:
minuit_projected = fitter.FitMinuit.init_from_covariance(
    projected_covariance,
    DataVel,
    parameter_dict,
    likelihood_type="multivariate_gaussian_projected",
)
minuit_projected.run()

[ 000001.78 ]: 11-05 07:46  root            INFO     ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 4088                       │              Nfcn = 273              │
│ EDM = 3.41e-05 (Goal: 0.0001)    │            time = 0.4 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬─────────┬───────────┬────────

{'fs8': 0.3758095211465344,
 'sigv': 0.44227115900005887,
 'alpha': 0.1272453497334782,
 'beta': 3.0936601019203263,
 'M_0': -19.06253330921393,
 'sigma_M': 0.14039425313645412}

Comparison: best fit, error and time of one likelihood call

In [7]:
def time_call(likelihood, x, repeat=20):
    likelihood(x)
    t0 = time.perf_counter()
    for _ in range(repeat):
        likelihood(x)
    return (time.perf_counter() - t0) / repeat


x = np.array(list(fiducial.values()))
for label, minuit_fitter in (("dense", minuit_dense), ("projected", minuit_projected)):
    values, errors = minuit_fitter.minuit.values, minuit_fitter.minuit.errors
    print(f"{label:10s} fs8 = {values['fs8']:.3f} ± {errors['fs8']:.3f}   "
          f"sigv = {values['sigv']:.0f} ± {errors['sigv']:.0f}   "
          f"{1e3 * time_call(minuit_fitter.likelihood, x):.2f} ms per call")

dense      fs8 = 0.375 ± 0.101   sigv = 0 ± 886   1.84 ms per call
projected  fs8 = 0.376 ± 0.102   sigv = 0 ± 916   1.31 ms per call
